# D131 — PEP 257: Python Docstring Conventions

A docstring explains how to use a Python module, package, class, function, or method.

**Goal:** write useful one-line and multi-line docstrings for every common kind of Python object.

## Why documentation belongs with code

Code tells Python **how** something works. Documentation helps people understand:

- what the object is for
- what callers must provide
- what is returned
- what can fail
- whether the operation changes state
- any rules that are not obvious from the signature

A docstring stays close to the code and is available through `help()`, `__doc__`, IDE tooltips, and documentation tools.

## What does PEP 257 cover?

**PEP 257** is the Python Enhancement Proposal for **Docstring Conventions**. It standardizes the high-level structure and purpose of docstrings, but it does not require a particular markup format.

Official reference: [PEP 257 — Docstring Conventions](https://peps.python.org/pep-0257/).

## What is a docstring?

A docstring is a string literal placed as the **first statement** in a module, function, class, or method. Python stores it in the object's `__doc__` attribute.

A string placed later in the body is just an unused string—it is not that object's main docstring.

In [ ]:
def calculate_subtotal(prices):
    """Return the sum of the supplied prices."""
    return sum(prices)


print(calculate_subtotal.__doc__)
assert calculate_subtotal([500, 750]) == 1250

## Docstring or comment?

Use a **docstring** to document an object's public purpose and usage. Use a `# comment` to explain a local implementation decision.

```python
def calculate_shipping(total):
    """Return the shipping charge for an order total."""
    # The courier contract makes delivery free from ₹3,000.
    return 0 if total >= 3000 else 120
```

The docstring is for the caller. The comment is for someone maintaining the calculation.

## Basic conventions

- Use `"""triple double quotes"""` consistently.
- Use `r"""raw triple double quotes"""` when the text contains backslashes.
- Write a one-line docstring only for a genuinely obvious case.
- End the summary with a period.
- For a function or method, use command form: `Return ...`, not `Returns ...`.
- Do not repeat the complete function signature inside the docstring.
- For a multi-line docstring, write a summary, a blank line, and then details.
- Put the closing quotes of a multi-line docstring on their own line.

## 1. One-line function docstring

Use this form when the behavior is obvious and needs no qualifications.

In [ ]:
def square(number):
    """Return the square of a number."""
    return number * number


assert square(5) == 25

Why this works:

- the summary fits on one line
- it begins with the action `Return`
- it describes the result
- it ends with a period
- the opening and closing quotes remain on the same line

## 2. Multi-line function docstring

Use a multi-line docstring when inputs, output rules, exceptions, side effects, or limitations are not obvious.

In [ ]:
def apply_discount(amount, percent=0):
    """Return an amount after applying a percentage discount.

    The percentage must be from 0 through 100. The function does
    not round the result, so the caller controls currency rounding.

    Args:
        amount: Original non-negative amount.
        percent: Discount percentage; defaults to 0.

    Returns:
        The amount remaining after the discount.

    Raises:
        ValueError: If amount is negative or percent is outside
            the inclusive range 0 through 100.
    """
    if amount < 0:
        raise ValueError("Amount cannot be negative")
    if not 0 <= percent <= 100:
        raise ValueError("Percent must be from 0 through 100")
    return amount * (1 - percent / 100)


assert apply_discount(1000, 10) == 900

The `Args`, `Returns`, and `Raises` headings above use a common Google-style layout. PEP 257 does **not** mandate these heading names; it requires a clear high-level structure. Select one detailed format for a project and use it consistently.

## What should a function docstring document?

Include only the parts that apply:

- behavior and purpose
- arguments and meaningful defaults
- return value or yielded values
- exceptions intentionally raised
- visible side effects, such as changing an object or writing a file
- restrictions or valid call states
- whether keyword arguments form part of the public interface

Do not explain obvious Python syntax line by line.

## 3. Function with no return value

Document the effect instead of inventing a return value.

In [ ]:
def mark_paid(order):
    """Change an order's status to paid.

    This function mutates the supplied dictionary in place.

    Args:
        order: Order dictionary containing a `status` key.
    """
    order["status"] = "paid"


sample_order = {"id": 101, "status": "pending"}
result = mark_paid(sample_order)
assert result is None
assert sample_order["status"] == "paid"

The important non-obvious fact is the **side effect**: the original dictionary changes. That matters more to a caller than stating that Python implicitly returns `None`.

## 4. Generator function docstring

A generator **yields** values over time. Describe what each iteration yields rather than saying it returns the completed collection.

In [ ]:
def iter_high_value_orders(orders, minimum_total):
    """Yield orders whose total meets the minimum.

    Args:
        orders: Iterable of dictionaries containing a `total` key.
        minimum_total: Inclusive lower limit for yielded orders.

    Yields:
        Each qualifying order in its original input order.
    """
    for order in orders:
        if order["total"] >= minimum_total:
            yield order


orders = [{"id": 1, "total": 500}, {"id": 2, "total": 2000}]
assert list(iter_high_value_orders(orders, 1000)) == [orders[1]]

## 5. Module docstring

A module docstring is the first statement in a `.py` file. It should summarize the module and normally list its public classes, exceptions, functions, and other exported objects.

```python
"""Calculate prices and taxes for customer orders.

Classes:
    InvoiceCalculator: Calculate invoice totals.

Functions:
    calculate_subtotal: Sum item amounts.
    calculate_tax: Calculate tax on a taxable amount.
"""

TAX_RATE = 0.18
```

Place the module docstring before imports and module constants.

## 6. Package docstring

A package is documented in its `__init__.py` module docstring. List the modules and subpackages exposed by the package.

```python
"""Provide e-commerce models, pricing, and file readers.

Modules:
    models: Product, order, and invoice domain objects.
    pricing: Discount, tax, and shipping calculations.
    readers: JSON and CSV input readers.
"""
```

This tells a user where to begin without requiring them to inspect the directory tree.

## 7. Script docstring

For a command-line program, the module docstring can double as a usage message. It should explain the command, syntax, arguments, environment variables, and files when relevant.

```python
"""Create an order summary from a JSON file.

Usage:
    python order_summary.py INPUT_JSON [--output OUTPUT_CSV]

Arguments:
    INPUT_JSON          Path to the source order file.
    --output OUTPUT_CSV Optional destination; defaults to summary.csv.
"""
```

A new user should be able to run the script from this documentation.

## 8. Class docstring

A class docstring summarizes what instances represent and documents public methods and important public instance variables. Leave a blank line between the class docstring and the first method.

In [ ]:
class ShoppingCart:
    """Collect purchasable items and calculate their subtotal.

    Attributes:
        items: Items currently stored in the cart.

    Public methods:
        add: Add an item to the cart.
        subtotal: Calculate the current item subtotal.
    """

    def __init__(self):
        """Initialize an empty shopping cart."""
        self.items = []

    def add(self, name, price):
        """Add a named item and its price to the cart."""
        self.items.append({"name": name, "price": price})

    def subtotal(self):
        """Return the sum of all item prices."""
        return sum(item["price"] for item in self.items)


cart = ShoppingCart()
cart.add("Keyboard", 1500)
assert cart.subtotal() == 1500

## 9. Constructor docstring

PEP 257 says public methods, including `__init__`, should have docstrings. Use the class docstring for the overall abstraction and the constructor docstring for initialization inputs and validation.

Avoid repeating the same paragraph in both places. In the previous example:

- `ShoppingCart` explains what the object represents.
- `ShoppingCart.__init__` explains the initial state.

## 10. Method docstring, including state changes

A method may have restrictions based on object state. Document those restrictions and the exception.

In [ ]:
class Order:
    """Represent a customer order that can be confirmed once."""

    def __init__(self, order_id):
        """Initialize a draft order with the supplied identifier."""
        self.order_id = order_id
        self.status = "draft"

    def confirm(self):
        """Confirm this draft order.

        Raises:
            RuntimeError: If the order is already confirmed.
        """
        if self.status == "confirmed":
            raise RuntimeError("Order is already confirmed")
        self.status = "confirmed"


order = Order("ORD-101")
order.confirm()
assert order.status == "confirmed"

## 11. Property docstring

A property looks like an attribute to its caller. Document the value it exposes, not the fact that it calls a getter.

In [ ]:
class OrderItem:
    """Represent one priced line in an order."""

    def __init__(self, quantity, unit_price):
        """Initialize an item with quantity and unit price."""
        self.quantity = quantity
        self.unit_price = unit_price

    @property
    def amount(self):
        """Return the quantity multiplied by the unit price."""
        return self.quantity * self.unit_price


item = OrderItem(2, 750)
assert item.amount == 1500
print(OrderItem.amount.__doc__)

## 12. Exception class docstring

Custom exception names normally end in `Error`. Its docstring states the condition represented by the exception.

In [ ]:
class InsufficientStockError(Exception):
    """Indicate that requested quantity exceeds available stock."""


def reserve_stock(requested, available):
    """Return remaining stock after reserving a quantity.

    Raises:
        InsufficientStockError: If requested exceeds available.
    """
    if requested > available:
        raise InsufficientStockError("Not enough stock")
    return available - requested


assert reserve_stock(2, 5) == 3

## 13. Inherited, extended, and overridden behavior

When most behavior is inherited, document how the subclass differs.

- **override**: the subclass replaces superclass behavior without calling it
- **extend**: the subclass calls superclass behavior and adds more work

In [ ]:
class BaseExporter:
    """Export records as text."""

    def export(self, records):
        """Return records converted to text."""
        return str(records)


class CsvExporter(BaseExporter):
    """Export records as comma-separated rows.

    This class overrides `export` to produce CSV instead of the
    base class's general text representation.
    """

    def export(self, records):
        """Return records joined as comma-separated rows."""
        return "\n".join(",".join(row) for row in records)


assert CsvExporter().export([["A", "1"]]) == "A,1"

## 14. Attribute docstrings

PEP 257 recognizes a string immediately after a simple assignment at module, class, or `__init__` level as an **attribute docstring**.

```python
DEFAULT_TAX_RATE = 0.18
"""Default tax rate applied to taxable order amounts."""


class Product:
    category = "general"
    """Category used when a product has no explicit category."""
```

This is less obvious: Python does not assign these strings to the value's `__doc__`. Documentation tools may extract them by reading source code.

## 15. Additional docstrings

A string immediately following another docstring is called an **additional docstring**.

```python
def calculate_total(items):
    """Return the total amount for all items."""
    """Items must contain numeric `amount` values."""
    return sum(item["amount"] for item in items)
```

Only the first string becomes `calculate_total.__doc__`. Some source-processing tools may combine or otherwise use the additional string. For ordinary application code, one well-structured docstring is usually clearer.

## 16. Raw docstrings containing backslashes

Use a raw triple-quoted docstring when examples contain Windows paths, regular expressions, or other backslashes. This prevents escape sequences such as `\n` from becoming special characters.

In [ ]:
def is_order_code(text):
    r"""Return whether text matches ``ORD\d+``.

    The pattern accepts `ORD` followed by one or more digits.
    """
    import re

    return re.fullmatch(r"ORD\d+", text) is not None


assert is_order_code("ORD101") is True
assert is_order_code("ORDER101") is False

## 17. Type hints and docstrings work together

Type hints describe machine-readable types. Docstrings describe meaning, valid ranges, units, behavior, and errors. Avoid needlessly repeating obvious type information.

In [ ]:
def calculate_tax(amount: float, rate: float = 0.18) -> float:
    """Return tax for a non-negative amount.

    Args:
        amount: Taxable amount in the caller's currency.
        rate: Tax rate as a decimal fraction, not a percentage.

    Raises:
        ValueError: If amount or rate is negative.
    """
    if amount < 0 or rate < 0:
        raise ValueError("Amount and rate cannot be negative")
    return amount * rate


assert calculate_tax(1000) == 180

The annotation says `rate` is a `float`. The docstring explains the non-obvious semantic rule: callers provide `0.18`, not `18`.

## 18. Accessing documentation

Python makes the main docstring available at runtime.

In [ ]:
import inspect


print(calculate_tax.__doc__)
print("--- cleaned docstring ---")
print(inspect.getdoc(calculate_tax))

`inspect.getdoc()` cleans uniform indentation. The built-in `help(calculate_tax)` also formats the signature and docstring through Python's documentation system.

From a terminal, documentation for an importable module can be displayed with:

```text
python -m pydoc module_name
```

`pydoc` imports the module, so module-level code executes during documentation discovery. Protect script-only execution with `if __name__ == "__main__":`.

## Weak and improved docstrings

### Repeats the signature

```python
# Weak
def find_product(product_id):
    """find_product(product_id) -> dict"""
```

Python can inspect the signature. Explain the behavior and important outcome instead:

```python
# Improved
def find_product(product_id):
    """Return the matching product, or `None` when it is absent."""
```

### Says nothing useful

```python
# Weak
def calculate_total(items):
    """Calculate total."""
```

```python
# Improved when the rules are not obvious
def calculate_total(items):
    """Return the sum of item amounts before tax and shipping."""
```

The improved version defines which total is calculated.

### Explains implementation instead of the contract

```python
# Weak
"""Loop through items, read amount, add it to total, and return total."""

# Improved
"""Return the sum of all item amounts."""
```

The caller needs the contract. The loop is visible in the source and may change later.

## Complete public API example

This example combines class, constructor, property, method, argument, return, exception, and state documentation.

In [ ]:
class Inventory:
    """Track available quantity by product SKU.

    Attributes:
        quantities: Current available quantity for each SKU.
    """

    def __init__(self, quantities=None):
        """Initialize inventory from an optional SKU-to-quantity mapping.

        A copy is stored so later changes to the caller's mapping do
        not change inventory state.
        """
        self.quantities = dict(quantities or {})

    @property
    def total_units(self):
        """Return the available quantity across all SKUs."""
        return sum(self.quantities.values())

    def reserve(self, sku, quantity):
        """Reserve units and return the remaining quantity.

        Args:
            sku: Identifier of the product to reserve.
            quantity: Positive number of units to reserve.

        Returns:
            Quantity remaining for the SKU after reservation.

        Raises:
            ValueError: If quantity is not positive.
            KeyError: If the SKU is unknown.
            InsufficientStockError: If quantity exceeds stock.

        Side effects:
            Decreases stored stock when the reservation succeeds.
        """
        if quantity <= 0:
            raise ValueError("Quantity must be positive")
        if sku not in self.quantities:
            raise KeyError("Unknown SKU")
        if quantity > self.quantities[sku]:
            raise InsufficientStockError("Not enough stock")

        self.quantities[sku] -= quantity
        return self.quantities[sku]


inventory = Inventory({"KEY-1": 5, "MOU-1": 3})
assert inventory.total_units == 8
assert inventory.reserve("KEY-1", 2) == 3
assert inventory.total_units == 6

## When a docstring is unnecessary

PEP 257 recommends docstrings for modules and exported/public functions, classes, methods, and constructors. Not every tiny private helper needs a long docstring.

```python
def _is_valid_sku(sku):
    return isinstance(sku, str) and bool(sku.strip())
```

If the private helper's name and implementation make the behavior completely obvious, a docstring may add noise. Document it when rules, side effects, or limitations are not obvious.

## Practice

Add a complete multi-line docstring to the function below. Document its purpose, arguments, return value, boundary rule, and exception.

Rules:

- `weight` is measured in kilograms.
- Weight must be greater than zero.
- Shipping costs ₹50 up to and including 5 kg.
- Shipping costs ₹100 above 5 kg.

In [ ]:
def shipping_fee(weight):
    # Add a PEP 257-style docstring as the first statement.
    if weight <= 0:
        raise ValueError("Weight must be positive")
    return 50 if weight <= 5 else 100


assert shipping_fee(5) == 50
assert shipping_fee(6) == 100

## Docstring review checklist

- [ ] The docstring is the first statement in the documented object.
- [ ] Triple double quotes are used.
- [ ] The summary is short, clear, and ends with a period.
- [ ] A function summary uses command form such as `Return`, `Yield`, or `Write`.
- [ ] A multi-line docstring has a blank line after its summary.
- [ ] Inputs include meaning, units, defaults, or valid ranges when needed.
- [ ] Return or yielded values are described.
- [ ] Intentional exceptions and call restrictions are documented.
- [ ] Side effects and state changes are clear.
- [ ] The docstring does not merely repeat the signature or source code.
- [ ] The project's chosen detailed format is used consistently.

## Recap

- PEP 257 defines high-level conventions for Python docstrings.
- The first string in a module, class, function, or method becomes `__doc__`.
- One-line docstrings suit obvious behavior; multi-line docstrings explain contracts and important details.
- Modules, packages, scripts, classes, constructors, methods, properties, generators, and exceptions need documentation suited to their role.
- Attribute and additional docstrings are source-level forms used by some documentation tools.
- Useful documentation explains meaning, behavior, failures, and side effects—not obvious syntax.